## **Task 1: Read All Datasets Using PySpark DataFrames**
Description:

The first step of the pipeline is to load the three raw CSV datasets (drivers.csv, trips.csv, and trip_logs.csv) into PySpark DataFrames. The datasets are read using spark.read.csv() with the header=True and inferSchema=True options to automatically detect column names and data types. After loading, the data is verified using show(), printSchema(), and count().

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Read the datasets
drivers_df = spark.read.csv(
    "/Volumes/workspace/default/ride_sharing_project/drivers.csv",
    header=True,
    inferSchema=True
)

trips_df = spark.read.csv(
    "/Volumes/workspace/default/ride_sharing_project/trips.csv",
    header=True,
    inferSchema=True
)

trip_logs_df = spark.read.csv(
    "/Volumes/workspace/default/ride_sharing_project/trip_logs.csv",
    header=True,
    inferSchema=True
)

# Display the datasets
display(drivers_df.limit(5))

display(trips_df.limit(5))

display(trip_logs_df.limit(5))

# Print schema
drivers_df.printSchema()
trips_df.printSchema()
trip_logs_df.printSchema()

# Record counts
print("Drivers:", drivers_df.count())
print("Trips:", trips_df.count())
print("Trip Logs:", trip_logs_df.count())

driver_id,name,city,rating
1,Rahul_1,Delhi,4.6
2,Priya_2,Mumbai,3.7
3,Rahul_3,Pune,3.6
4,Sneha_4,Delhi,3.5
5,Priya_5,Mumbai,4.3


trip_id,driver_id,pickup_location,drop_location,distance_km,fare_amount,trip_status
1,78,IT Park,IT Park,8.6,0.0,Cancelled
2,114,Airport,Mall,17.54,203.08,Completed
3,132,Railway Station,Mall,17.27,213.02,Completed
4,58,Railway Station,IT Park,20.55,185.6,Completed
5,19,IT Park,IT Park,12.47,177.11,Completed


log_id,trip_id,start_time,end_time,delay_minutes,cancellation_flag
1,1,2025-01-03T01:44:00.000Z,null,0,1
2,2,2025-01-02T04:34:00.000Z,2025-01-02T04:50:00.000Z,20,0
3,3,2025-01-06T22:55:00.000Z,2025-01-06T23:29:00.000Z,18,0
4,4,2025-01-07T22:14:00.000Z,2025-01-07T22:47:00.000Z,18,0
5,5,2025-01-05T00:15:00.000Z,2025-01-05T01:15:00.000Z,20,0


root
 |-- driver_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- rating: double (nullable = true)

root
 |-- trip_id: integer (nullable = true)
 |-- driver_id: integer (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- drop_location: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- trip_status: string (nullable = true)

root
 |-- log_id: integer (nullable = true)
 |-- trip_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- delay_minutes: integer (nullable = true)
 |-- cancellation_flag: integer (nullable = true)

Drivers: 150
Trips: 150
Trip Logs: 150


## **Task 2: Store Raw Data into Bronze Layer**
Description:

The Bronze layer stores the raw data exactly as received without applying any transformations. Each dataset is written in Parquet format, which provides efficient storage and faster query performance. This layer acts as the raw data repository for the pipeline.

In [0]:
bronze_path = "/Volumes/workspace/default/ride_sharing_project/bronze"

drivers_df.write.mode("overwrite").parquet(f"{bronze_path}/drivers")

trips_df.write.mode("overwrite").parquet(f"{bronze_path}/trips")

trip_logs_df.write.mode("overwrite").parquet(f"{bronze_path}/trip_logs")

Verification:

In [0]:
# Read Bronze Layer Data
bronze_drivers = spark.read.parquet(f"{bronze_path}/drivers")
bronze_trips = spark.read.parquet(f"{bronze_path}/trips")
bronze_logs = spark.read.parquet(f"{bronze_path}/trip_logs")

# Display Bronze Tables
display(bronze_drivers.limit(5))

display(bronze_trips.limit(5))

display(bronze_logs.limit(5))

# Display Record Counts
print("Bronze Drivers:", bronze_drivers.count())
print("Bronze Trips:", bronze_trips.count())
print("Bronze Trip Logs:", bronze_logs.count())

driver_id,name,city,rating
1,Rahul_1,Delhi,4.6
2,Priya_2,Mumbai,3.7
3,Rahul_3,Pune,3.6
4,Sneha_4,Delhi,3.5
5,Priya_5,Mumbai,4.3


trip_id,driver_id,pickup_location,drop_location,distance_km,fare_amount,trip_status
1,78,IT Park,IT Park,8.6,0.0,Cancelled
2,114,Airport,Mall,17.54,203.08,Completed
3,132,Railway Station,Mall,17.27,213.02,Completed
4,58,Railway Station,IT Park,20.55,185.6,Completed
5,19,IT Park,IT Park,12.47,177.11,Completed


log_id,trip_id,start_time,end_time,delay_minutes,cancellation_flag
1,1,2025-01-03T01:44:00.000Z,null,0,1
2,2,2025-01-02T04:34:00.000Z,2025-01-02T04:50:00.000Z,20,0
3,3,2025-01-06T22:55:00.000Z,2025-01-06T23:29:00.000Z,18,0
4,4,2025-01-07T22:14:00.000Z,2025-01-07T22:47:00.000Z,18,0
5,5,2025-01-05T00:15:00.000Z,2025-01-05T01:15:00.000Z,20,0


Bronze Drivers: 150
Bronze Trips: 150
Bronze Trip Logs: 150


## **Task 3: Perform Joins Between Drivers, Trips, and Trip Logs**
Description:

In the Silver layer, the three datasets are joined to create a unified dataset. The Trips dataset is joined with the Drivers dataset using the driver_id column. The resulting DataFrame is then joined with the Trip Logs dataset using the trip_id column. This combined dataset contains driver details, trip information, and trip log information, which is used for further cleaning, transformations, and analytics.

In [0]:
# Join Trips with Drivers
joined_df = trips_df.join(
    drivers_df,
    on="driver_id",
    how="inner"
)

# Join the result with Trip Logs
joined_df = joined_df.join(
    trip_logs_df,
    on="trip_id",
    how="inner"
)

# Display the joined dataset
display(joined_df)

# Print schema
joined_df.printSchema()

# Record count
print("Total Joined Records:", joined_df.count())

trip_id,driver_id,pickup_location,drop_location,distance_km,fare_amount,trip_status,name,city,rating,log_id,start_time,end_time,delay_minutes,cancellation_flag
1,78,IT Park,IT Park,8.6,0.0,Cancelled,Anjali_78,Delhi,5.0,1,2025-01-03T01:44:00.000Z,null,0,1
2,114,Airport,Mall,17.54,203.08,Completed,Neha_114,Bangalore,3.7,2,2025-01-02T04:34:00.000Z,2025-01-02T04:50:00.000Z,20,0
3,132,Railway Station,Mall,17.27,213.02,Completed,Priya_132,Hyderabad,4.0,3,2025-01-06T22:55:00.000Z,2025-01-06T23:29:00.000Z,18,0
4,58,Railway Station,IT Park,20.55,185.6,Completed,Priya_58,Bangalore,4.1,4,2025-01-07T22:14:00.000Z,2025-01-07T22:47:00.000Z,18,0
5,19,IT Park,IT Park,12.47,177.11,Completed,Rahul_19,Hyderabad,3.9,5,2025-01-05T00:15:00.000Z,2025-01-05T01:15:00.000Z,20,0
6,103,Airport,IT Park,7.61,95.83,Completed,Vikas_103,Bangalore,3.7,6,2025-01-04T22:19:00.000Z,2025-01-04T22:33:00.000Z,14,0
7,57,Airport,City Center,6.05,0.0,Cancelled,Priya_57,Pune,4.6,7,2025-01-02T02:38:00.000Z,null,0,1
8,64,IT Park,City Center,23.1,204.42,Completed,Amit_64,Mumbai,3.6,8,2025-01-06T08:45:00.000Z,2025-01-06T09:30:00.000Z,9,0
9,144,City Center,City Center,15.7,0.0,Cancelled,Rahul_144,Bangalore,4.4,9,2025-01-04T11:00:00.000Z,null,0,1
10,110,IT Park,IT Park,21.1,249.72,Completed,Vikas_110,Mumbai,4.5,10,2025-01-01T23:20:00.000Z,2025-01-01T23:47:00.000Z,16,0


root
 |-- trip_id: integer (nullable = true)
 |-- driver_id: integer (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- drop_location: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- log_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- delay_minutes: integer (nullable = true)
 |-- cancellation_flag: integer (nullable = true)

Total Joined Records: 150


## **Task 4: Clean Data by Handling Null Values and Incorrect Records**
Description

The raw data is cleaned by removing duplicate records and handling null values. Driver ratings are validated to ensure they fall between 0 and 5. This step improves data quality before performing analytics.

In [0]:
# Remove duplicate records
drivers = bronze_drivers.dropDuplicates()
trips = bronze_trips.dropDuplicates()
trip_logs = bronze_logs.dropDuplicates()

# Handle null values
drivers = drivers.na.drop()
trips = trips.na.drop()
trip_logs = trip_logs.na.drop()

# Validate driver ratings
drivers = drivers.filter(
    (col("rating") >= 0) &
    (col("rating") <= 5)
)

display(drivers.limit(5))
display(trips.limit(5))
display(trip_logs.limit(5))

driver_id,name,city,rating
14,Rohit_14,Pune,3.7
17,Rahul_17,Delhi,4.5
25,Anjali_25,Mumbai,4.5
38,Sneha_38,Pune,5.0
43,Neha_43,Hyderabad,3.5


trip_id,driver_id,pickup_location,drop_location,distance_km,fare_amount,trip_status
1,78,IT Park,IT Park,8.6,0.0,Cancelled
30,31,Mall,City Center,20.31,242.94,Completed
46,71,Railway Station,IT Park,3.05,0.0,Cancelled
52,116,Mall,City Center,22.9,0.0,Cancelled
64,150,IT Park,Mall,12.03,164.81,Completed


log_id,trip_id,start_time,end_time,delay_minutes,cancellation_flag
15,15,2025-01-02T08:39:00.000Z,2025-01-02T09:23:00.000Z,19,0
73,73,2025-01-03T14:31:00.000Z,2025-01-03T15:27:00.000Z,19,0
113,113,2025-01-04T01:41:00.000Z,2025-01-04T02:22:00.000Z,13,0
148,148,2025-01-02T06:33:00.000Z,2025-01-02T07:24:00.000Z,13,0
23,23,2025-01-06T11:01:00.000Z,2025-01-06T11:12:00.000Z,10,0


## **Task 5: Create Derived Columns**
Description

New columns are created to enrich the dataset. Trip duration is calculated using the start and end timestamps, and a completion flag is created based on the trip status.

In [0]:
joined_df = bronze_trips.join(
    bronze_drivers,
    on="driver_id",
    how="inner"
)

joined_df = joined_df.join(
    bronze_logs,
    on="trip_id",
    how="inner"
)

joined_df = joined_df.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp("end_time") -
        unix_timestamp("start_time")
    ) / 60
)

joined_df = joined_df.withColumn(
    "completion_flag",
    when(col("trip_status") == "Completed", 1).otherwise(0)
)

display(joined_df.select(
    "trip_id",
    "trip_duration_minutes",
    "completion_flag"
))

trip_id,trip_duration_minutes,completion_flag
1,null,0
2,16.0,1
3,34.0,1
4,33.0,1
5,60.0,1
6,14.0,1
7,null,0
8,45.0,1
9,null,0
10,27.0,1


## **Task 6: Filter Invalid Data**
Description

Invalid records such as negative trip distances, negative fares, and missing timestamps are removed to ensure reliable analytics.

In [0]:
joined_df = joined_df.filter(col("distance_km") >= 0)

joined_df = joined_df.filter(col("fare_amount") >= 0)

joined_df = joined_df.filter(
    col("start_time").isNotNull() &
    col("end_time").isNotNull()
)

print("Valid Records:", joined_df.count())

Valid Records: 63


## **Task 7: Write Cleaned Data into Silver Layer**
Description

The cleaned and transformed dataset is stored in the Silver layer in Parquet format.

In [0]:
silver_path = "/Volumes/workspace/default/ride_sharing_project/silver"

joined_df.write.mode("overwrite").parquet(
    f"{silver_path}/final_data"
)

silver_df = spark.read.parquet(
    f"{silver_path}/final_data"
)

display(silver_df.limit(5))

print("Silver Records:", silver_df.count())

trip_id,driver_id,pickup_location,drop_location,distance_km,fare_amount,trip_status,name,city,rating,log_id,start_time,end_time,delay_minutes,cancellation_flag,trip_duration_minutes,completion_flag
2,114,Airport,Mall,17.54,203.08,Completed,Neha_114,Bangalore,3.7,2,2025-01-02T04:34:00.000Z,2025-01-02T04:50:00.000Z,20,0,16.0,1
3,132,Railway Station,Mall,17.27,213.02,Completed,Priya_132,Hyderabad,4.0,3,2025-01-06T22:55:00.000Z,2025-01-06T23:29:00.000Z,18,0,34.0,1
4,58,Railway Station,IT Park,20.55,185.6,Completed,Priya_58,Bangalore,4.1,4,2025-01-07T22:14:00.000Z,2025-01-07T22:47:00.000Z,18,0,33.0,1
5,19,IT Park,IT Park,12.47,177.11,Completed,Rahul_19,Hyderabad,3.9,5,2025-01-05T00:15:00.000Z,2025-01-05T01:15:00.000Z,20,0,60.0,1
6,103,Airport,IT Park,7.61,95.83,Completed,Vikas_103,Bangalore,3.7,6,2025-01-04T22:19:00.000Z,2025-01-04T22:33:00.000Z,14,0,14.0,1


Silver Records: 63


## **Task 8: Generate Driver Performance Metrics**
Description

Driver performance is evaluated using total trips, average rating, and total revenue earned.

In [0]:
driver_performance = silver_df.groupBy(
    "driver_id",
    "name"
).agg(
    count("trip_id").alias("total_trips"),
    avg("rating").alias("average_rating"),
    sum("fare_amount").alias("total_revenue")
)

display(driver_performance)

driver_id,name,total_trips,average_rating,total_revenue
87,Vikas_87,1,3.6,236.14
137,Karan_137,1,3.7,28.19
63,Amit_63,1,4.6,334.21
11,Vikas_11,1,3.6,134.96
132,Priya_132,2,4.0,387.74
19,Rahul_19,1,3.9,177.11
110,Vikas_110,1,4.5,249.72
86,Arjun_86,1,3.6,110.15
85,Karan_85,1,5.0,157.98
31,Neha_31,1,4.6,242.94


## **Task 9: Calculate Cancellation Rate per Driver**
Description

The cancellation rate is calculated as the percentage of cancelled trips for each driver.

In [0]:
cancellation_rate = silver_df.groupBy(
    "driver_id"
).agg(
    (
        sum("cancellation_flag") /
        count("*") * 100
    ).alias("cancellation_rate")
)

display(cancellation_rate)

driver_id,cancellation_rate
18,0.0
38,0.0
64,0.0
16,0.0
105,0.0
94,0.0
31,0.0
5,0.0
77,0.0
111,0.0


## **Task 10: Identify High-Demand Pickup Locations**
Description

Pickup locations are analyzed to determine areas with the highest ride demand.

In [0]:
high_demand_locations = silver_df.groupBy(
    "pickup_location"
).count().orderBy(
    desc("count")
)

display(high_demand_locations)

pickup_location,count
IT Park,17
Railway Station,14
Mall,13
Airport,13
City Center,6


## **Task 11: Generate Revenue-Related Insights**
Description

Revenue generated by each driver is calculated using the total fare amount collected.

In [0]:
revenue_insights = silver_df.groupBy(
    "driver_id"
).agg(
    sum("fare_amount").alias("total_revenue")
)

display(revenue_insights)

driver_id,total_revenue
18,250.87
38,223.44
64,204.42
16,211.15
105,122.72
94,204.54
31,242.94
5,39.54
77,189.57
111,55.27


Delay Analysis

In [0]:
from pyspark.sql.functions import avg, max

delay_analysis = joined_df.groupBy("city").agg(
    avg("delay_minutes").alias("avg_delay_minutes"),
    max("delay_minutes").alias("max_delay_minutes")
)

delay_analysis.show()
delay_analysis.write.mode("overwrite").parquet(
    f"{gold_path}/delay_analysis"
)

+---------+------------------+-----------------+
|     city| avg_delay_minutes|max_delay_minutes|
+---------+------------------+-----------------+
|    Delhi|12.277777777777779|               20|
|Hyderabad|12.714285714285714|               20|
|     Pune|              10.7|               19|
|Bangalore|               9.6|               20|
|   Mumbai|11.923076923076923|               19|
+---------+------------------+-----------------+



## **Task 12: Store Aggregated Results in Gold Layer**
Description

The aggregated analytical tables are stored in the Gold layer in Parquet format for business reporting.

In [0]:
gold_path = "/Volumes/workspace/default/ride_sharing_project/gold"

driver_performance.write.mode("overwrite").parquet(
    f"{gold_path}/driver_performance"
)

cancellation_rate.write.mode("overwrite").parquet(
    f"{gold_path}/cancellation_rate"
)

high_demand_locations.write.mode("overwrite").parquet(
    f"{gold_path}/high_demand_locations"
)

revenue_insights.write.mode("overwrite").parquet(
    f"{gold_path}/revenue_insights"
)

## **Task 13: Validate Data Consistency Across Layers**
Description

Record counts are validated across Bronze, Silver, and Gold layers to ensure successful data processing.

In [0]:
print("Bronze Drivers:", bronze_drivers.count())
print("Bronze Trips:", bronze_trips.count())
print("Bronze Trip Logs:", bronze_logs.count())

print("Silver Records:", silver_df.count())

print("Driver Performance:", driver_performance.count())
print("Cancellation Rate:", cancellation_rate.count())
print("High Demand Locations:", high_demand_locations.count())
print("Revenue Insights:", revenue_insights.count())

Bronze Drivers: 150
Bronze Trips: 150
Bronze Trip Logs: 150
Silver Records: 63
Driver Performance: 50
Cancellation Rate: 50
High Demand Locations: 5
Revenue Insights: 50


## **Task 14: Apply Optimization Techniques in Spark**
Description

Spark optimization techniques help improve query performance. Since Databricks Serverless does not support caching, the execution plan is analyzed using explain(True) to understand how Spark optimizes the query execution.

In [0]:
optimized_df = silver_df.repartition(4)

optimized_df.explain(True)

print("Optimized Records:", optimized_df.count())

silver_df.explain(True)

== Parsed Logical Plan ==
Repartition 4, true, false, 0, Unknown
+- Relation [trip_id#11465,driver_id#11466,pickup_location#11467,drop_location#11468,distance_km#11469,fare_amount#11470,trip_status#11471,name#11472,city#11473,rating#11474,log_id#11475,start_time#11476,end_time#11477,delay_minutes#11478,cancellation_flag#11479,trip_duration_minutes#11480,completion_flag#11481] parquet

== Analyzed Logical Plan ==
trip_id: int, driver_id: int, pickup_location: string, drop_location: string, distance_km: double, fare_amount: double, trip_status: string, name: string, city: string, rating: double, log_id: int, start_time: timestamp, end_time: timestamp, delay_minutes: int, cancellation_flag: int, trip_duration_minutes: double, completion_flag: int
Repartition 4, true, false, 0, Unknown
+- Relation [trip_id#11465,driver_id#11466,pickup_location#11467,drop_location#11468,distance_km#11469,fare_amount#11470,trip_status#11471,name#11472,city#11473,rating#11474,log_id#11475,start_time#11476,end

## **Task 15: Rank Drivers Using Window Functions**
Description

A window function is used to rank drivers based on total revenue, enabling identification of the best-performing drivers.

In [0]:
window_spec = Window.orderBy(
    desc("total_revenue")
)

ranked_drivers = driver_performance.withColumn(
    "driver_rank",
    rank().over(window_spec)
)

display(ranked_drivers)

ranked_drivers.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/ride_sharing_project/gold/ranked_drivers"
)

print("Ranked Drivers:", ranked_drivers.count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


driver_id,name,total_trips,average_rating,total_revenue,driver_rank
114,Neha_114,3,3.7000000000000006,720.44,1
107,Rahul_107,2,3.9,530.96,2
65,Amit_65,3,3.6,493.75,3
102,Vikas_102,2,4.8,390.52,4
132,Priya_132,2,4.0,387.74,5
76,Priya_76,3,4.2,369.47,6
20,Vikas_20,2,4.1,336.66,7
63,Amit_63,1,4.6,334.21,8
112,Amit_112,1,4.5,326.4,9
103,Vikas_103,2,3.7,256.07,10


Ranked Drivers: 50


## **Final Validation Cell**

In [0]:
print("Ride-Sharing Analytics & Driver Performance Pipeline")

print("Bronze Drivers:", bronze_drivers.count())
print("Bronze Trips:", bronze_trips.count())
print("Bronze Trip Logs:", bronze_logs.count())

print("Silver Records:", silver_df.count())

print("Driver Performance Records:", driver_performance.count())

print("Cancellation Rate Records:", cancellation_rate.count())

print("High Demand Locations:", high_demand_locations.count())

print("Revenue Insights:", revenue_insights.count())

print("Delay Analysis Records:", delay_analysis.count())

print("Ranked Drivers:", ranked_drivers.count())

Ride-Sharing Analytics & Driver Performance Pipeline
Bronze Drivers: 150
Bronze Trips: 150
Bronze Trip Logs: 150
Silver Records: 63
Driver Performance Records: 50
Cancellation Rate Records: 50
High Demand Locations: 5
Revenue Insights: 50
Delay Analysis Records: 5


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Ranked Drivers: 50


# **Conclusion**

- Successfully implemented a Medallion Architecture (Bronze, Silver, Gold).
- Cleaned and transformed raw ride-sharing data.
- Generated business KPIs including driver performance, cancellation rate, delay analysis, demand hotspots, and revenue insights.
- Applied Spark optimization techniques and window functions for scalable analytics.
- The pipeline supports efficient, data-driven decision-making for ride-sharing operations.